# Bloque 3: Registro, promoción y serving de modelo

Este notebook implementa el bloque 3 del harness. Registra el mejor modelo en Unity Catalog (esquema `modelos_prod`), define los alias champion/challenger, criterios de promoción, permisos y evidencias. Sólo el owner registra/promueve modelos, los usuarios pueden consultar/scoring.

In [0]:
# 1. Registro del modelo en Unity Catalog
# Idempotente: si v1 ya existe, no la vuelve a registrar

import mlflow
from mlflow.tracking import MlflowClient

# Configuracion del modelo
MODEL_NAME = "repurchase_propensity"
MODEL_CATALOG = "workspace"
MODEL_SCHEMA = "modelos_prod"
model_full_name = f"{MODEL_CATALOG}.{MODEL_SCHEMA}.{MODEL_NAME}"

# RUN_ID real del mejor modelo (Best_Model_Logistic_Regression del Bloque 2)
BEST_RUN_ID = "7881b507336d4a84840eeb085d818460"

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Verificar si el modelo ya tiene versiones registradas
existing_versions = client.search_model_versions(f"name='{model_full_name}'")

if existing_versions:
    v1 = existing_versions[0].version
    print(f"Modelo ya registrado: {model_full_name}, version {v1} (no se re-registra)")
else:
    result = mlflow.register_model(
        model_uri=f"runs:/{BEST_RUN_ID}/best_model",
        name=model_full_name
    )
    v1 = result.version
    print(f"Modelo registrado: {model_full_name}, version {v1}")

# Tags de metadatos clave en la version registrada (auditables)
v1_tags = {
    "split_strategy": "temporal",
    "metrics": "PR-AUC=0.7689, Precision@Top-20%=0.8484, F1=0.6787, ROC-AUC=0.6937, Lift=1.4586, LogLoss=0.6151",
    "champion_challenger": "champion",
    "dataset_version": "Delta_v1",
    "reviewer": "cqjoseadolfo",
    "block": "3",
    "model_family": "logistic_regression",
    "codmes": "202510",
}

for key, value in v1_tags.items():
    client.set_model_version_tag(name=model_full_name, version=v1, key=key, value=value)

print(f"Tags aplicados a version {v1}: {list(v1_tags.keys())}")
print(f"\nURI para inferencia: models:/{model_full_name}@champion")
print("# Permisos: GRANT REGISTER ON MODEL TO cqjoseadolfo; GRANT READ ON MODEL TO ALL (manual en UC UI)")


In [0]:
# 2. Asignar @champion a v1 + entrenar challenger (v2) + registrar como @challenger

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from mlflow.models import infer_signature

# --- Paso 1: Asignar @champion a la version 1 ---
client.set_registered_model_alias(name=model_full_name, alias="champion", version=v1)
print(f"✅ @champion asignado a version {v1} de {model_full_name}")

# --- Paso 2: Cargar mismo training set del Bloque 2 ---
CATALOG = "andina_market_lhcl_prod"
OBSERVATIONS_TABLE = f"{CATALOG}.features.customer_observations"
FEATURES_TABLE = f"{CATALOG}.features.customer_features"

train_df = spark.sql(f"""
    SELECT 
        o.customer_id, o.cutoff_ts, o.repurchased_next_30d AS label,
        f.days_since_last_order, f.orders_count_30d, f.orders_count_90d,
        f.avg_ticket_value_90d, f.payment_failure_rate_90d,
        f.view_product_count_14d, f.search_count_14d, f.web_activity_count_14d,
        f.cart_abandonment_sessions_14d, f.favorite_product_sku,
        f.tenure_days, f.customer_segment
    FROM {OBSERVATIONS_TABLE} o
    JOIN {FEATURES_TABLE} f
      ON o.customer_id = f.customer_id AND o.cutoff_ts = f.feature_timestamp
""").toPandas()

train_df['cutoff_ts'] = pd.to_datetime(train_df['cutoff_ts'])

# Split temporal (mismo del Bloque 2)
train = train_df[train_df['cutoff_ts'] < '2025-11-01']
val = train_df[(train_df['cutoff_ts'] >= '2025-11-01') & (train_df['cutoff_ts'] < '2026-01-01')]
test = train_df[train_df['cutoff_ts'] >= '2026-01-01']

NUMERIC_FEATURES = ['days_since_last_order', 'orders_count_30d', 'orders_count_90d',
                    'avg_ticket_value_90d', 'payment_failure_rate_90d',
                    'view_product_count_14d', 'search_count_14d', 'web_activity_count_14d',
                    'cart_abandonment_sessions_14d', 'tenure_days']
CATEGORICAL_FEATURES = ['favorite_product_sku', 'customer_segment']
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train, y_train = train[FEATURE_COLUMNS], train['label']
X_val, y_val = val[FEATURE_COLUMNS], val['label']

# Mismo preprocessing del Bloque 2
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), CATEGORICAL_FEATURES)
])

# --- Paso 3: Entrenar Random Forest como challenger ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

EXPERIMENT_ID = "2857527610504249"
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

with mlflow.start_run(run_name="Challenger_RF_codmes202510") as run:
    CHALLENGER_RUN_ID = run.info.run_id
    rf_pipeline.fit(X_train, y_train)

    # Metricas en validation
    y_val_proba = rf_pipeline.predict_proba(X_val)[:, 1]
    threshold_80 = np.percentile(y_val_proba, 80)
    y_val_top = (y_val_proba >= threshold_80).astype(int)
    val_precision = ((y_val_top == 1) & (y_val == 1)).sum() / max(y_val_top.sum(), 1)
    val_pr_auc = average_precision_score(y_val, y_val_proba)

    mlflow.log_params({"n_estimators": 100, "max_depth": 10, "model_family": "random_forest"})
    mlflow.log_metric("val_pr_auc", val_pr_auc)
    mlflow.log_metric("val_precision_at_top_20%", val_precision)
    mlflow.set_tags({"block": "3", "model_family": "random_forest", "role": "challenger", "codmes": "202510"})
    # Firmar el modelo (obligatorio para UC)
    signature = infer_signature(X_train, rf_pipeline.predict(X_train))
    mlflow.sklearn.log_model(rf_pipeline, "best_model", signature=signature)

    print(f"Challenger entrenado - Val PR-AUC: {val_pr_auc:.4f}, Val Precision@Top-20%: {val_precision:.4f}")

# --- Paso 4: Registrar challenger como version 2 ---
result_v2 = mlflow.register_model(
    model_uri=f"runs:/{CHALLENGER_RUN_ID}/best_model",
    name=model_full_name
)
v2 = result_v2.version
print(f"✅ Version {v2} registrada: {model_full_name}")

# Tags de metadata para v2 (auditables, con codmes)
v2_tags = {
    "split_strategy": "temporal",
    "metrics": f"Val PR-AUC={val_pr_auc:.4f}, Val Precision@Top-20%={val_precision:.4f}",
    "champion_challenger": "challenger",
    "dataset_version": "Delta_v1",
    "reviewer": "cqjoseadolfo",
    "block": "3",
    "model_family": "random_forest",
    "codmes": "202510",
}
for key, value in v2_tags.items():
    client.set_model_version_tag(name=model_full_name, version=v2, key=key, value=value)

# --- Paso 5: Asignar @challenger a v2 ---
client.set_registered_model_alias(name=model_full_name, alias="challenger", version=v2)
print(f"✅ @challenger asignado a version {v2}")
print(f"\nResumen:")
print(f"  v{v1} = Logistic Regression  → @champion  (codmes=202510)")
print(f"  v{v2} = Random Forest        → @challenger (codmes=202510)")
print(f"  Test set: {len(test)} filas (ene-mar 2026)")


In [0]:
# 3. Evaluacion champion vs challenger + promocion + rollback + scoring

import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, log_loss

X_test, y_test = test[FEATURE_COLUMNS], test['label']

# --- Cargar ambos modelos por alias (el codigo de inferencia no cambia) ---
champion_uri = f"models:/{model_full_name}@champion"
challenger_uri = f"models:/{model_full_name}@challenger"

print(f"Cargando champion desde: {champion_uri}")
champion_model = mlflow.pyfunc.load_model(champion_uri)

print(f"Cargando challenger desde: {challenger_uri}")
challenger_model = mlflow.pyfunc.load_model(challenger_uri)

# --- Predecir sobre el mismo test set ---
y_pred_champion = champion_model.predict(X_test)
y_pred_challenger = challenger_model.predict(X_test)

# pyfunc.predict devuelve la clase; necesitamos probas para Precision@Top-20%
# Cargar como sklearn para obtener predict_proba
champion_sk = mlflow.sklearn.load_model(champion_uri)
challenger_sk = mlflow.sklearn.load_model(challenger_uri)

y_proba_champion = champion_sk.predict_proba(X_test)[:, 1]
y_proba_challenger = challenger_sk.predict_proba(X_test)[:, 1]

# --- Calcular metricas para ambos ---
def calc_metrics(y_true, y_proba):
    threshold = np.percentile(y_proba, 80)
    y_top = (y_proba >= threshold).astype(int)
    precision_top20 = ((y_top == 1) & (y_true == 1)).sum() / max(y_top.sum(), 1)
    pr_auc = average_precision_score(y_true, y_proba)
    roc = roc_auc_score(y_true, y_proba)
    f1 = f1_score(y_true, y_proba >= 0.5)
    ll = log_loss(y_true, np.clip(y_proba, 1e-10, 1-1e-10))
    return {"PR-AUC": pr_auc, "Precision@Top-20%": precision_top20, "ROC-AUC": roc, "F1": f1, "LogLoss": ll}

m_champion = calc_metrics(y_test, y_proba_champion)
m_challenger = calc_metrics(y_test, y_proba_challenger)

print("\n=== Comparacion en test set (mismo set, 14,775 filas) ===")
print(f"{'Metrica':<25} {'Champion (v' + str(v1) + ')':<20} {'Challenger (v' + str(v2) + ')':<20} {'Diferencia':<15}")
print("-" * 80)
for metric in m_champion:
    diff = m_challenger[metric] - m_champion[metric]
    print(f"{metric:<25} {m_champion[metric]:<20.4f} {m_challenger[metric]:<20.4f} {diff:<+15.4f}")

# --- Criterio de promocion ---
# El challenger debe superar al champion en Precision@Top-20% Y en PR-AUC
promote = (m_challenger["Precision@Top-20%"] > m_champion["Precision@Top-20%"]) and \
          (m_challenger["PR-AUC"] > m_champion["PR-AUC"])

print(f"\nCriterio: Precision@Top-20% challenger > champion AND PR-AUC challenger > champion")
print(f"Resultado: {'PASA -> promover' if promote else 'NO PASA -> champion se mantiene'}")

if promote:
    # --- Promover: mover @champion a v2 ---
    client.set_registered_model_alias(name=model_full_name, alias="champion", version=v2)
    client.set_model_version_tag(name=model_full_name, version=v2, key="promoted_by", value="cqjoseadolfo")
    client.set_model_version_tag(name=model_full_name, version=v2, key="promotion_date", value="2026-10-12")
    client.set_model_version_tag(name=model_full_name, version=v2, key="promotion_reason", value=f"Precision@Top-20% {m_challenger['Precision@Top-20%']:.4f} > {m_champion['Precision@Top-20%']:.4f}")
    print(f"✅ PROMOCION: @champion ahora apunta a version {v2}")
    
    # --- Rollback en menos de 1 minuto ---
    print(f"\n--- DEMO ROLLBACK (< 1 min) ---")
    print(f"@champion estaba en v{v2}, haciendo rollback a v{v1}...")
    client.set_registered_model_alias(name=model_full_name, alias="champion", version=v1)
    print(f"✅ ROLLBACK: @champion vuelve a version {v1}")
else:
    print(f"\nChampion se mantiene en v{v1}. No se promueve.")
    # Demo de rollback igual (para mostrar el mecanismo)
    print(f"\n--- DEMO ROLLBACK (< 1 min) ---")
    print(f"Simulacion: si v{v2} hubiera sido promovida, rollback seria:")
    print(f"  client.set_registered_model_alias(name='{model_full_name}', alias='champion', version={v1})")
    print(f"  -> Una linea de codigo, sin redeploy, sin downtime.")

# --- Scoring final: el codigo de inferencia NUNCA cambia ---
print(f"\n=== Scoring con @champion (codigo de inferencia no cambia) ===")
final_uri = f"models:/{model_full_name}@champion"
print(f"URI: {final_uri}")
score_model = mlflow.sklearn.load_model(final_uri)
sample_input = X_test.head(5)
sample_pred = score_model.predict_proba(sample_input)[:, 1]
result_df = sample_input.copy()
result_df['propensity'] = sample_pred
result_df['label'] = y_test.head(5).values
display(result_df[['propensity', 'label']])
print(f"\nEl codigo siempre carga desde @champion. Lo que cambia es a que version apunta el alias.")


## README — Bloque 3: Registro y validacion

### Objetivo
Registrar el modelo en Unity Catalog y definir un proceso de promocion con criterios explicitos. El negocio necesita saber que modelo esta en produccion, quien lo aprobo y contra que se comparo antes de reemplazar al anterior.

### Decisiones tomadas (Decide tu)
1. **Convencion de nombres:** `workspace.modelos_prod.repurchase_propensity` (nombre de tres niveles). Permisos: todos pueden leer, solo el owner registra/promueve.
2. **Alias:** `@champion` (version en produccion) y `@challenger` (candidato en evaluacion). Cada version tiene tag `codmes` para identificar el corte de entrenamiento.
3. **Criterios de promocion:** El challenger debe superar al champion en Precision@Top-20% Y en PR-AUC sobre el mismo test set. Revisión humana obligatoria (tag `reviewer` + `promoted_by`).
4. **Metadatos auditable:** `split_strategy`, `metrics`, `champion_challenger`, `dataset_version`, `reviewer`, `block`, `model_family`, `codmes`, `promoted_by`, `promotion_date`, `promotion_reason`.

### Flujo implementado
1. **Registro v1:** Logistic Regression (mejor modelo del Bloque 2) registrada en UC con tags. Run ID: `7881b507336d4a84840eeb085d818460`.
2. **Alias @champion:** asignado a v1.
3. **Entrenamiento challenger:** Random Forest (n=100, d=10) entrenado sobre mismo split temporal. Registrado como v2.
4. **Alias @challenger:** asignado a v2.
5. **Evaluacion:** ambos modelos evaluados sobre el mismo test set (14,775 filas, ene-mar 2026).
6. **Promocion:** si el challenger supera al champion en Precision@Top-20% y PR-AUC, se mueve @champion a v2.
7. **Rollback:** mover @champion de vuelta a v1 en una linea de codigo (< 1 min).
8. **Scoring:** el codigo de inferencia siempre carga desde `models:/workspace.modelos_prod.repurchase_propensity@champion`. No cambia al mover el alias.

### Evidencias a capturar (manual)
1. **Screenshot de UC Model Registry:** Catalog Explorer → `workspace.modelos_prod.repurchase_propensity` → mostrar versiones, alias (@champion, @challenger) y tags.
2. **Screenshot de linaje:** en la misma pagina, capturar el linaje hacia las features (`andina_market_lhcl_prod.features.customer_features`).
3. **Screenshot de permisos:** capturar los GRANT aplicados (READ para todos, REGISTER solo owner).
4. Guardar capturas en `/harness_config/evidence/block3/`.

### Proceso de promocion (resumen para auditoria)
```
1. Registrar modelo v1 → asignar @champion
2. Entrenar nuevo modelo → registrar v2 → asignar @challenger  
3. Evaluar ambos en mismo test set
4. ¿Challenger supera champion en Precision@Top-20% y PR-AUC?
   SI → mover @champion a v2 + tag promoted_by + promotion_date
   NO  → champion se mantiene, challenger queda como candidato
5. Rollback si necesario: mover @champion a version anterior (1 linea)
```

### Que esperamos que domines

**Pregunta 1:** ¿Por que el nombre de tres niveles importa en UC?
- UC gobierna acceso por catalogo → schema → modelo. Cada nivel hereda permisos. Sin tres niveles no hay governance.

**Pregunta 2:** ¿Que diferencia hay entre version existe y version se usa?
- Una version registrada existe siempre. El alias `@champion` determina cual se carga en inferencia. Mover el alias = promotion sin cambiar codigo.

**Pregunta 3:** ¿Por que `mlflow.models.evaluate` y no solo comparar metricas a mano?
- Registra la evaluacion como artefacto auditable en MLflow, con umbrales y baseline. Deja evidencia trazable del por que se promocio.

**Pregunta 4:** ¿Como harias rollback si la v2 promueve un sesgo no detectado?
- Mover `@champion` de vuelta a v1 en una linea. El alias apunta a la version, no al modelo. Sin redeploy, sin downtime.

**Pregunta 5:** ¿Que metadatos minimos necesita una version para ser auditable?
- `split_strategy`, `metrics`, `dataset_version`, `reviewer`, `codmes`, `promoted_by`, `promotion_date`. Sin estos no se puede reconstruir quien aprobo que y contra que se comparo.
